# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This project is primarily a ranking and scoring task. Each content page will receive a review-priority score, and pages will be ordered from highest to lowest priority. The purpose is not to automatically decide that a page must be edited, but to help an SEO specialist or content editor decide which pages should be inspected first. A classification model may be used internally to estimate decline risk, but the final business output is a ranked review queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The provisional proxy target in the starter dataset is whether a page is currently classified as declining. The proxy is derived from the observed change in impressions between the previous 30-day period and the most recent 30-day period. It does not measure whether refreshing the page would be successful. It only provides a temporary signal for ranking pages that may deserve human review.

For the later capstone, a stronger target should use separate time windows: features from an earlier period and an observed performance outcome from a later period.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric will be Precision@50. It measures the proportion of pages labelled as declining by the provisional proxy among the top 50 pages recommended for review

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd

# Load the starter dataset
data_url = (
    "https://raw.githubusercontent.com/"
    "Suhail-Ahmed7/flyrank-ml-internship-suhail/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# Select the columns relevant to our ML lane
lane_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "trend_direction",
]

# Keep pages with at least 100 impressions
lane_df = df.loc[
    df["impressions_90d"] >= 100,
    lane_columns
].copy()

# Create the provisional target column
# 1 = declining page
# 0 = not declining
lane_df["is_declining_label"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print(f"Rows in lane slice: {len(lane_df):,}")
print("Unit of analysis: one row = one pseudonymized content page")

# Show the real dataframe
display(lane_df.head(10))

# Show the target column clearly
display(
    lane_df[
        ["content_id", "trend_direction", "is_declining_label"]
    ].head(10)
)

# Show how many pages belong to each target class
target_summary = (
    lane_df["is_declining_label"]
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Not declining",
        1: "Declining"
    })
    .to_frame("count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(lane_df) * 100
).round(2)

display(target_summary)

Rows in lane slice: 22,006
Unit of analysis: one row = one pseudonymized content page


,content_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,keyword article,187,20,3803,29,0.76,10.6,17,5.88,down,1
1,content_a1fb4e703a9e,keyword article,445,25,15320,7,0.05,20.3,9,0.00,down,1
2,content_9aa793d4d895,keyword article,141,20,12581,11,0.09,36.5,11,0.00,down,1
3,content_331d6c4de07b,keyword article,463,22,11751,58,0.49,6.2,78,1.28,stable,0
4,content_d99b7a2d90ca,keyword article,263,14,19140,24,0.13,44.0,145,0.00,down,1
5,content_d4084a4bc775,keyword article,147,20,3970,1,0.03,8.5,5,0.00,down,1
7,content_a63219c6e95a,keyword article,445,22,1724,1,0.06,21.2,28,3.57,stable,0
8,content_5e6c160719bc,keyword article,90,20,32574,29,0.09,46.0,68,5.88,down,1
9,content_c27558df2b0c,keyword article,257,104,1240,2,0.16,4.9,3,0.00,down,1
10,content_d8ee6cc6d642,keyword article,329,104,20919,324,1.55,2.2,326,6.75,stable,0


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1
10,content_d8ee6cc6d642,stable,0


,count,percentage
is_declining_label,,
Not declining,8854,40.23
Declining,13152,59.77


One row represents one pseudonymized content page. The is_declining_label column is a provisional target: 1 represents a declining page and 0 represents a page that is not declining. The trend_direction column is used only to construct the target and will not be used as a model feature because that would cause target leakage.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A transparent fixed rule will be used as the baseline, for example prioritizing pages that are old, visible, and declining. However, one rule may not work equally well for every content type, position range, traffic level, or engagement pattern. A page with low CTR at position 3 is different from a page with the same CTR at position 18. Similarly, content age may matter differently depending on impressions, recent movement, and engagement.

Machine learning may help combine these interacting signals and produce a more useful priority ordering. It earns its place only if it performs better than the transparent baseline and still provides understandable reason codes. The output will remain decision support for a human reviewer rather than an automatic editing decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.